[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_DSP/Filter_Design.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Filter Design

The application-focused companion to [Foundations of Signal Processing](./Foundations_of_Signal_Processing_1.ipynb): there we learned *what* the frequency domain is; here we learn to *sculpt* it. By the end you will design FIR and IIR filters, read their responses, and apply them to real signals.

## 0. Introduction

A filter is a system that treats different frequencies differently — keep the heartbeat, drop the power-line hum; keep the voice, drop the hiss. Digital filters come in two families with a classic trade-off:

| | FIR | IIR |
|---|---|---|
| Impulse response | finite | infinite (feedback) |
| Always stable? | **yes** | no — poles must stay in the unit circle |
| Exactly linear phase? | can be | no |
| Order needed for sharp cutoff | high | **low** |


## 1. Pre-requisites

- [Intro to Python](../Intro_Programming/Intro_Python/Intro_Python.ipynb) — NumPy & Matplotlib.
- [Foundations of Signal Processing](./Foundations_of_Signal_Processing_1.ipynb) — convolution, DTFT/DFT, the $z$-plane (Sessions 2–3 especially).

We use `scipy.signal` throughout — install with `conda install scipy` or `pip install scipy`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

fs = 1000  # sampling rate [Hz] used throughout

def plot_response(b, a=1, fs=fs, title=""):
    """Magnitude (dB) and phase of a digital filter."""
    w, h = signal.freqz(b, a, worN=2048, fs=fs)
    fig, (ax0, ax1) = plt.subplots(2, 1, figsize=(8, 4.5), sharex=True)
    ax0.plot(w, 20 * np.log10(np.maximum(np.abs(h), 1e-8)))
    ax0.set_ylabel("magnitude [dB]"); ax0.set_ylim(-100, 5); ax0.grid(True)
    ax0.set_title(title)
    ax1.plot(w, np.unwrap(np.angle(h)))
    ax1.set_ylabel("phase [rad]"); ax1.set_xlabel("frequency [Hz]"); ax1.grid(True)
    plt.tight_layout(); plt.show()

---
### 🕐 Session 1 of 3 — *FIR Filters* (~35 min)
**Goal:** design a windowed-sinc FIR low-pass, read its response, and understand linear phase.
**Builds on:** [Foundations](./Foundations_of_Signal_Processing_1.ipynb) Sessions 3 & 6. &nbsp; **Feeds into:** Session 2 (IIR).

---

## 2. FIR Filters

💡 **Intuition.** An FIR filter is nothing but a **weighted moving average**: the output is a fixed set of weights (the *taps*) slid across the signal — a convolution. The ideal low-pass in frequency is a rectangle, whose time-domain shape is the infinite `sinc`; a practical FIR filter is that sinc *truncated and tapered by a window*. More taps ⇒ closer to the rectangle ⇒ sharper cutoff.

### 2.1. The Windowed-Sinc Design

`firwin` does exactly the recipe above: sample the ideal sinc, multiply by a window (Hamming by default), normalize.

In [ ]:

# YOUR CODE HERE


**What just happened.** The taps are a **sinc**: a tall centre spike, then decaying oscillations symmetric about the middle, tapered smoothly to zero at both ends. `firwin` did not invent a filter — it sampled the ideal low-pass's impulse response and windowed it.

**Why a sinc, specifically.** The *ideal* low-pass is a rectangle in frequency: 1 in the passband, 0 in the stopband, with a vertical edge. Inverse-transform a rectangle and you get a sinc — infinite in both directions and non-causal, so the perfect filter would need infinitely many taps and knowledge of the infinite future. Unbuildable. Truncating it to 101 taps and tapering the ends with a Hamming window is exactly what makes it realisable, and **every artifact in the frequency response traces back to that truncation**. The stopband ripples are the price of the truncation; the finite transition width is the price of the taper.

**Note the symmetry**, because it is not decorative. The taps read the same forwards and backwards, and that symmetry is *precisely* the condition for linear phase. It is why `numtaps` is odd here — an odd length gives a genuine centre tap and a group delay of exactly $(101-1)/2 = 50$ samples, an integer. Break the symmetry and linear phase is gone.

**And the tap count is the design dial.** More taps means a longer truncation, so a closer approximation to the ideal rectangle, so a sharper cutoff. The cost is delay: group delay is $(N-1)/2$ samples, so 101 taps at 1 kHz costs 50 ms and a 1001-tap filter would cost half a second. **Sharpness is bought with latency**, always — which is why a real-time system cannot simply add taps until the response looks good.

Notice too that this filter is a *moving average with clever weights*. Convolution slides these 101 numbers across the signal and sums; there is no other mechanism. Everything in the next cell's frequency response is a consequence of these 101 values.

In [ ]:

# YOUR CODE HERE


Things to read off the plot:

- **Passband** (0–100 Hz): flat at 0 dB — frequencies pass untouched.
- **Transition band**: the rolloff around the cutoff; narrower needs more taps.
- **Stopband**: ripples bounded by the window's sidelobe level (Hamming ⇒ ~−53 dB).
- **Phase**: a perfectly straight line — *linear phase*.

💡 **Intuition.** Linear phase means **every frequency is delayed by the same amount** ($\frac{N-1}{2}$ samples — half the filter length). The waveform's shape survives; it just arrives late. Nonlinear phase smears different frequencies by different delays, *distorting the shape even when magnitudes are untouched* — fatal for waveforms you must interpret (ECG, seismic, communications symbols).

### 2.2. Windows Trade Ripple for Width

The window choice is a knob: low sidelobes (less stopband ripple) cost a wider mainlobe (slower rolloff) — the uncertainty principle from [Foundations Session 4](./Foundations_of_Signal_Processing_1.ipynb) wearing a hard hat.

In [ ]:

# YOUR CODE HERE


**What just happened.** Three windows, identical cutoff and identical tap count — and the responses trade off in exactly opposite directions. **Boxcar** (no taper at all) has the steepest rolloff and the worst stopband, with sidelobes only about −21 dB. **Blackman–Harris** pushes sidelobes below −90 dB and pays with a visibly wider transition band. **Hamming** sits between, at roughly −53 dB.

**This is the uncertainty principle, in filter-design clothing.** A window that is abrupt in time — the boxcar simply stops — is spread out in frequency, producing large sidelobes. A window that tapers gently to zero is smoother in time and therefore more concentrated in frequency, giving small sidelobes; but the taper effectively shortens the useful filter length, which widens the mainlobe and slows the rolloff. Same tension as [Foundations Session 4](./Foundations_of_Signal_Processing_1.ipynb), same mathematics, now with engineering units attached.

**Which means there is no best window, only a best window for a requirement.** Ask what each is for:

- Rejecting a strong out-of-band interferer 80 dB down? You need the stopband depth — Blackman–Harris, and accept the wider transition.
- Separating two closely-spaced bands with nothing hostile nearby? You need the sharp transition — boxcar or Hamming, and tolerate the sidelobes.

The design question is never "which window is good" but "which artifact can I afford."

**And the third knob is tap count.** Windows trade sidelobe level against transition width *at fixed length*; adding taps improves the transition width without changing the sidelobe level, since sidelobes are a property of the window shape rather than of $N$. So a practical recipe falls out: pick the window from the stopband requirement, then pick the tap count from the transition-width requirement, then check the resulting group delay is affordable. That ordering is worth remembering — it is how the design actually gets done.

---
### 🕐 Session 2 of 3 — *IIR Filters* (~35 min)
**Goal:** design Butterworth/Chebyshev filters; understand stability from pole locations.
**Builds on:** Session 1; [Foundations](./Foundations_of_Signal_Processing_1.ipynb) Session 2 (Laplace). &nbsp; **Feeds into:** Session 3 (implementation).

---

## 3. IIR Filters

💡 **Intuition.** An IIR filter adds **feedback**: the output is a mix of recent inputs *and recent outputs*. Feedback lets a handful of coefficients ring like a resonator, so an IIR filter of order 4 can cut as sharply as an FIR of order 100. The price: the ringing must die out — every pole must sit **inside the unit circle** — and phase is no longer linear.

### 3.1. The Classical Families

Each classical design answers "what should the passband and stopband look like?" differently:

- **Butterworth** — maximally flat passband, gentle rolloff.
- **Chebyshev I** — ripples in the passband, buys a faster rolloff.
- **Elliptic** — ripples in both bands, fastest rolloff of all.

(All are analog prototypes mapped to digital via the *bilinear transform* — the Laplace $s$-plane from [Foundations Session 2](./Foundations_of_Signal_Processing_1.ipynb) bent onto the $z$-plane's unit circle.)

In [ ]:

# YOUR CODE HERE


**What just happened.** Three **order-4** filters — nine coefficients each — achieving rolloffs comparable to Session 1's **101-tap** FIR. That is roughly a 25× reduction in arithmetic per sample, and it is what feedback buys.

The mechanism is worth stating plainly: an IIR filter's output depends on recent *outputs* as well as recent inputs. That feedback lets four poles ring like a resonator, producing an effectively long impulse response from very few coefficients. An FIR filter has to store every value of its impulse response explicitly; an IIR filter *generates* one.

**Read the three families as three answers to one question** — what should the passband and stopband look like?

- **Butterworth** is maximally flat in the passband, with no ripple anywhere, and pays with the gentlest rolloff of the three.
- **Chebyshev I** shows visible ripple in the passband and buys a noticeably faster transition with it.
- **Elliptic** ripples in *both* bands and achieves the steepest rolloff available at this order.

There is no winner. Each spends the same budget — order 4 — on a different priority, and the right choice depends on whether your application tolerates passband ripple (audio usually can; a precision measurement often cannot) and how sharp a transition you actually need.

**But do not let the 25× stand alone, because the comparison is not free.** Two costs come with feedback. First, **stability**: the ringing must decay, which requires every pole strictly inside the unit circle — an IIR filter *can* be unstable, while an FIR filter cannot be, ever. Second, **phase**: none of these has the straight-line phase of Session 1's FIR, so waveform shape distorts even where the magnitude response is flat. For an ECG, where the diagnosis lives in the shape of the complex, that alone can disqualify an IIR design.

So the honest summary is a trade rather than a ranking: IIR is ~25× cheaper with a stability condition and shape distortion; FIR is expensive, unconditionally stable, and shape-preserving. Compute-constrained real-time work leans IIR; offline shape-sensitive work leans FIR. The next cell makes the stability condition visible.

### 3.2. Poles, Zeros & Stability

The transfer function $H(z)$ is a ratio of polynomials; its **zeros** pin the response down (notches) and its **poles** push it up (resonance). Stability = all poles strictly inside the unit circle.

In [ ]:

# YOUR CODE HERE


**What just happened.** Four poles, all comfortably inside the unit circle, with `max |pole|` printed in the title and an `assert` enforcing $|p| < 1$. The filter is stable, and the plot is the *proof* rather than an illustration.

**Poles and zeros, in one sentence each.** Zeros pull the response *down* — a zero on the unit circle produces an exact null at that frequency, which is how notch filters work. Poles push the response *up* — a pole near the unit circle produces a resonant peak at the corresponding angle. The magnitude response at frequency $\omega$ is essentially the product of distances to the zeros divided by the product of distances to the poles, evaluated at $e^{j\omega}$. Once students hold that picture, they can *read* a pole-zero plot into a frequency response without computing anything, which is a genuinely useful skill.

**Stability is one inequality, and it has a physical meaning.** A pole at radius $r$ contributes a term decaying like $r^n$. If $r < 1$ that dies out; if $r > 1$ it grows without bound, and the filter's own output feeds an explosion. This is exactly the geometric-series dichotomy from [Sequences & Series](../Intro_Math/Analysis/Numerical_Sequences_and_Series.ipynb) and precisely the same boundary as the vanishing/exploding gradient in [RNNs](../Intro_Time_Series/Intro_RNN.ipynb) — a decaying impulse response and a vanishing gradient are the same fact viewed from opposite ends. FIR filters have no poles at all, which is *why* they are unconditionally stable; there is nothing that can grow.

**And the practically important consequence.** Watch what happens as a pole approaches the unit circle: the resonance sharpens, the ringing lasts longer, and the filter becomes increasingly sensitive to small changes in its coefficients. That sensitivity is not hypothetical. A high-order IIR filter expressed in direct `(b, a)` form has coefficients spanning many orders of magnitude, and rounding them — in float32, or in fixed point on an embedded target — can move a pole across the unit circle and turn a working filter into an oscillator.

That is why Session 3 insists on **second-order sections**. Factoring the filter into biquads keeps each stage's poles independent, so a rounding error perturbs one small factor instead of a high-degree polynomial whose roots are wildly sensitive to its coefficients. Same filter mathematically, vastly better numerically — and it is the same "factor, don't expand" lesson that appears in [RLS](../Intro_Time_Series/Intro_RLS.ipynb)'s square-root forms.

---
### 🕐 Session 3 of 3 — *Filters in Practice* (~40 min)
**Goal:** clean a real (synthetic) signal end-to-end; avoid the classic implementation pitfalls.
**Builds on:** Sessions 1–2. &nbsp; **Feeds into:** [Adaptive Filtering](../Intro_Time_Series/README.md) — filters that tune themselves.

---

## 4. Filters in Practice

### 4.1. The Scenario

An ECG-like signal contaminated by two enemies: 60 Hz power-line hum and broadband noise. Plan: a **notch** for the hum, a **low-pass** for the hiss.

In [ ]:
# synthetic "ECG": periodic sharp pulses + baseline wander

# YOUR CODE HERE


**What just happened.** The measured trace is visibly corrupted — a fast regular oscillation riding on top of the pulses, plus a general fuzz — while the truth underneath is a clean train of sharp beats on a slow wandering baseline.

**Diagnose before designing, because the two contaminants need opposite tools.** The 60 Hz hum is a *single known frequency*, so it occupies one narrow spot in the spectrum. The broadband noise is spread across *all* frequencies. Those call for different instruments: a **notch** for the hum, surgical and nearly free elsewhere; a **low-pass** for the hiss, blunt and unavoidably costly.

Ask what would happen if you tried to remove the hum with a low-pass instead. You would need a cutoff below 60 Hz — and the ECG's diagnostic value is in the *sharpness* of its pulses, which is exactly the high-frequency content such a filter would destroy. You would remove the interference and the signal together. **Match the tool to the shape of the interference in frequency**, and do that before writing any design code.

**Note the three components deliberately planted here**, because each exercises something different. The `gausspulse` beats are the sharp features that only survive a filter with good high-frequency behaviour and honest phase. The 0.3 Hz `baseline` wander is slow drift — real in ECG, caused by breathing and electrode movement, and *not* removed by anything in this workshop, since it sits below every cutoff used. And the hum is at exactly 60 Hz, the North American mains frequency, which is why 60 Hz notches are standard equipment in biomedical instrumentation (and 50 Hz in most of the rest of the world).

That baseline wander is worth flagging as an honest omission: the filtered result below still carries it, because no stage targets it. Removing it would need a *high*-pass around 0.5 Hz, which is exactly what clinical ECG front-ends do. The demo cleans two of the three problems and leaves the third, which is a fair reflection of how real pipelines are assembled one stage at a time.

### 4.2. Notch + Low-pass

In [ ]:
# 60 Hz notch (narrow IIR band-stop)
# 40 Hz FIR low-pass for the broadband noise

# YOUR CODE HERE


**What just happened.** RMSE **0.385 → 0.043**, a factor of 8.9, and the filtered trace tracks the truth closely enough that the two curves are hard to tell apart.

**Two stages, two targets.** `iirnotch(60, Q=30)` places a zero essentially *on* the unit circle at 60 Hz — a surgical null that removes the hum while leaving everything else nearly untouched. That is Session 2's pole-zero picture doing exactly what it promised: a zero on the circle is an exact null at that frequency. Then the 201-tap FIR low-pass at 40 Hz removes broadband noise above the ECG's useful band. Each contaminant got the tool shaped for it.

**The `filtfilt` detail is why the traces line up.** `filtfilt` runs the filter forwards and then backwards over the signal. The backward pass applies the exact conjugate phase of the forward pass, so all phase distortion cancels and the result is **zero-phase** — no delay at all, which is why the filtered curve sits on top of the truth rather than shifted right by the group delay.

That is a genuinely useful trick, and it comes with a hard restriction: the backward pass needs the *entire signal, including the future*. `filtfilt` is offline-only. A bedside ECG monitor cannot use it and must run `lfilter` causally, accepting delay and phase distortion. Ask which constraint applies before reaching for it — the phase-perfect result here is available precisely because nothing is real-time.

Note also that `filtfilt` applies the filter *twice*, so the effective magnitude response is squared: the actual attenuation is double the design figure in dB, and the effective order is doubled. Convenient, and worth knowing when you compare a design's response against what you measure.

**Two honest caveats on the 0.043.** First, **edge transients**: a filter takes roughly one filter length to warm up, so the first ~200 samples are untrustworthy, and the RMSE is computed over the whole signal including them — the number is slightly pessimistic, and a real pipeline would discard the warm-up. Second, and more substantively, the 0.3 Hz **baseline wander is still present** in the output. Nothing in this chain targets it, since it sits far below every cutoff used. Removing it needs a high-pass around 0.5 Hz, which is standard in clinical ECG front-ends.

So the residual 0.043 is not all noise — a real part of it is a contaminant we never attempted to remove. That is a fair picture of how filter chains get built: one stage per identified problem, and the residual tells you which problems remain.

### 4.3. Pitfalls Worth Their Own Slide

- **`lfilter` vs `filtfilt`** — `lfilter` runs causally (real-time capable) but delays and phase-distorts; `filtfilt` runs forward *and* backward, canceling all phase distortion — offline only, since it needs the future.
- **Edge transients** — a filter needs time to "warm up"; distrust the first ~one filter length of output.
- **Numerical fragility** — high-order IIR filters in `(b, a)` form can explode from rounding. Use **second-order sections**: factor the filter into biquads with `output="sos"` + `sosfiltfilt`.

In [ ]:
# The professional habit: SOS form for anything beyond order ~4
# small differences: two different filters approximating the same job

# YOUR CODE HERE


**What just happened.** A 10th-order Butterworth in second-order-section form produces an output differing from the 201-tap FIR result by at most **0.016** on this signal.

**Read that number for what it is, because it is easy to over-interpret.** These are **two different filters** — an order-10 IIR and a 201-tap FIR — designed to the same 40 Hz cutoff. The comment in the cell says so. A discrepancy of 0.016 therefore tells you the two designs agree closely on this particular signal; it says **nothing** about the numerical advantage of SOS form, which is what the surrounding discussion is about.

**What SOS actually buys, and why the demo does not show it.** The real comparison would be the *same* filter expressed two ways: direct `(b, a)` coefficients versus second-order sections. At order 10, the direct form's polynomial coefficients span many orders of magnitude, and the roots of a high-degree polynomial are notoriously sensitive to perturbations in its coefficients — round them in float32 or in fixed point on an embedded target and a pole can migrate across the unit circle, turning a working filter into an oscillator. SOS factors the filter into independent biquads, so a rounding error perturbs one small quadratic instead of a degree-10 polynomial. Mathematically identical filter, dramatically better conditioning.

Demonstrating that properly takes one extra experiment: build the same high-order filter in both forms, apply each, and compare — ideally at order 12 or beyond, where direct form visibly misbehaves. That would be a genuine SOS-vs-direct comparison, and it is a good five-minute exercise. As written, the cell establishes the *habit* without measuring the failure it prevents, and it is worth being clear about which of those you have seen.

**The rule to take away regardless: anything beyond about order 4, use `output="sos"`.** It costs nothing and removes an entire class of failure. And note the lineage — this is the same "factor, don't expand" principle as square-root Kalman and QR-based [RLS](../Intro_Time_Series/Intro_RLS.ipynb) updates. Whenever a computation involves a quantity that must stay inside a boundary (a pole inside the unit circle, a covariance positive-definite), work with factors rather than the expanded object, because factored forms cannot silently violate the constraint.

**The recipe this workshop leaves you with.** Waveform shape matters and you are offline? FIR with `filtfilt`, tap count set by the transition width you need. Compute-constrained and real-time? IIR in SOS form, and check the poles. Single-frequency interference? Notch it. And always look at magnitude *and* phase before trusting any filter — Session 1's linear-phase argument means a perfect magnitude response is not, on its own, evidence of a good filter.

## 5. Conclusion

Design recipe to take home:

1. Waveform shape matters / offline? → **FIR** (+ `filtfilt`), taps set by transition width.
2. Compute-constrained / real-time? → **IIR** in **SOS** form, check the poles.
3. Single-frequency interference? → **notch**.
4. Always *look* at magnitude **and** phase before trusting a filter.

---
## Where next

- [Adaptive Filtering (APA / Kalman)](../Intro_Time_Series/README.md) — when the interference won't hold still, the filter must learn.
- [Foundations of Signal Processing](./Foundations_of_Signal_Processing_1.ipynb) Session 6 — how convolution with long filters is computed fast.
- [Intro to FPGA](../Intro_FPGA/README.md) — implementing these filters as literal hardware.